<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/gradient_checkpointing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate datasets

In [ ]:
import torch
import time
import gc
import pandas as pd


from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
def run_hf_gradient_checkpointing_demo(
    model_name="gpt2-medium",
    gradient_checkpointing=False,
    batch_size=1,
    seq_len=512,
    steps=10
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
    model.train()

    if gradient_checkpointing:
      model.gradient_checkpointing_enable()
      model.config.use_cache=False

    else:
      model.gradient_checkpointing_disable()
      model.config.use_cache = True

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    input_ids = torch.randint(
        low=0,
        high=tokenizer.vocab_size,
        size=(batch_size, seq_len),
        device="cuda"
    )

    attention_mask = torch.ones_like(input_ids)
    labels = input_ids.clone()

    optimizer.zero_grad(set_to_none=True)
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    torch.cuda.synchronize()

    start_time = time.time()

    for step in range(steps):
      optimizer.zero_grad(set_to_none=True)

      outputs = model(
          input_ids = input_ids,
          attention_mask=attention_mask,
          labels=labels
      )

      loss = outputs.loss
      loss.backward()
      optimizer.step()

    torch.cuda.synchronize()

    end_time = time.time()

    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
    seconds_per_step = (end_time - start_time) / steps
    tokens_per_second = (batch_size * seq_len) / seconds_per_step

    result = {
        "gradient_checkpointing": gradient_checkpointing,
        "model": model_name,
        "batch_size": batch_size,
        "seq_len": seq_len,
        "peak_memory_GB": round(peak_memory_gb, 2),
        "seconds_per_step": round(seconds_per_step, 3),
        "tokens_per_second": round(tokens_per_second, 1),
    }

    del model, optimizer, input_ids, attention_mask, labels
    gc.collect()
    torch.cuda.empty_cache()

    return result





In [ ]:
no_ckpt = run_hf_gradient_checkpointing_demo(
    model_name="gpt2-medium",
    gradient_checkpointing=False,
    batch_size=2,
    seq_len=1024,
    steps=10
)
no_ckpt

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'gradient_checkpointing': False,
 'model': 'gpt2-medium',
 'batch_size': 2,
 'seq_len': 1024,
 'peak_memory_GB': 11.26,
 'seconds_per_step': 1.784,
 'tokens_per_second': 1148.1}

In [ ]:
with_ckpt = run_hf_gradient_checkpointing_demo(
    model_name="gpt2-medium",
    gradient_checkpointing=True,
    batch_size=2,
    seq_len=1024,
    steps=10
)

with_ckpt

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

{'gradient_checkpointing': True,
 'model': 'gpt2-medium',
 'batch_size': 2,
 'seq_len': 1024,
 'peak_memory_GB': 7.01,
 'seconds_per_step': 2.608,
 'tokens_per_second': 785.1}

In [ ]:
df = pd.DataFrame([no_ckpt, with_ckpt])
df

,gradient_checkpointing,model,batch_size,seq_len,peak_memory_GB,seconds_per_step,tokens_per_second
0,False,gpt2-medium,2,1024,11.26,1.784,1148.1
1,True,gpt2-medium,2,1024,7.01,2.608,785.1
